# UCS420: Cognitive Computing
## Assignment 4 – A Cognitive FAQ System Using Pandas (Nova 2.0)
Pre-requisite: Pandas DataFrame

**Name:** Prashant Narayan Verma  
**Roll No.:** 1024160140

## Q1: Build your personalized knowledge base
The last two digits of my roll number decide the categories of the 2 personalised entries: `category = ["billing", "account", "general"][d % 3]`.

In [1]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

NAME = "Prashant Narayan Verma"
ROLL_NUMBER = "1024160140"

categories = ["billing", "account", "general"]
last_two = [int(d) for d in ROLL_NUMBER[-2:]]
my_categories = [categories[d % 3] for d in last_two]

print("Roll number     :", ROLL_NUMBER)
print("Last two digits :", last_two)
for d, c in zip(last_two, my_categories):
    print(f"  digit {d} -> {d} % 3 = {d % 3} -> categories[{d % 3}] = '{c}'")

Roll number     : 1024160140
Last two digits : [4, 0]
  digit 4 -> 4 % 3 = 1 -> categories[1] = 'account'
  digit 0 -> 0 % 3 = 0 -> categories[0] = 'billing'


In [2]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# One realistic question + answer + 3 keywords for each possible category
personal_bank = {
    "account": {"question": "how do i change my registered email address",
                "answer": "Go to Profile > Account Settings > Email and confirm the change using the link sent to the new address.",
                "keywords": "email address update"},
    "billing": {"question": "how do i download my fee receipt",
                "answer": "Open Payments > History and click Download next to the transaction.",
                "keywords": "receipt download invoice"},
    "general": {"question": "where is your office located",
                "answer": "Our office is in Block B, First Floor, next to the library.",
                "keywords": "location address office"},
}

personal_entries = []
for c in my_categories:
    entry = dict(personal_bank[c])
    entry["category"] = c
    personal_entries.append(entry)

df = pd.DataFrame(fixed_entries + personal_entries)
print("Final knowledge base (6 rows):")
df

Final knowledge base (6 rows):


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i change my registered email address,Go to Profile > Account Settings > Email and confirm the change using the li...,email address update,account
5,how do i download my fee receipt,Open Payments > History and click Download next to the transaction.,receipt download invoice,billing


## Q2: Generate and score a hypothesis
Each FAQ entry is a *hypothesis* about what the user means. A query word that hits the entry's **keywords** is worth 2 points and a hit on the **question** text is worth 1 point. Confidence = score / (maximum possible score = 3 × number of query words), capped at 1.

In [3]:
STOPWORDS = {"what", "is", "the", "how", "do", "i", "my", "to", "can", "a", "an",
             "of", "for", "are", "your", "in", "on", "me", "you", "does"}
EMPTY = pd.DataFrame(columns=["question", "answer", "category", "score", "confidence"])

def score_query(query, df):
    """Return every matching FAQ entry, ranked by score (best first)."""
    q_words = set(query.lower().split()) - STOPWORDS
    if not q_words:
        return EMPTY.copy()

    rows = []
    for _, r in df.iterrows():
        kw_hits = len(q_words & set(r["keywords"].lower().split()))
        q_hits = len(q_words & (set(r["question"].lower().split()) - STOPWORDS))
        score = 2 * kw_hits + q_hits
        if score > 0:
            rows.append({"question": r["question"], "answer": r["answer"],
                         "category": r["category"], "score": score,
                         "confidence": round(min(score / (3 * len(q_words)), 1.0), 2)})
    if not rows:
        return EMPTY.copy()
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

In [4]:
for q in ["what is the fee", "reset my password", "download receipt", "pizza"]:
    print(f"Query: '{q}'")
    display(score_query(q, df))

Query: 'what is the fee'


,question,answer,category,score,confidence
0,what is the annual fee,The annual fee is Rs 500.,billing,3,1.00
1,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,3,1.00
2,how do i download my fee receipt,Open Payments > History and click Download next to the transaction.,billing,1,0.33


Query: 'reset my password'


,question,answer,category,score,confidence
0,how to reset password,Go to Settings > Reset Password.,account,6,1.0


Query: 'download receipt'


,question,answer,category,score,confidence
0,how do i download my fee receipt,Open Payments > History and click Download next to the transaction.,billing,6,1.0


Query: 'pizza'


,question,answer,category,score,confidence


## Q3: `same_category(category_name, df)`

In [5]:
def same_category(category_name, df):
    """Return all questions that belong to the given category."""
    return df.loc[df["category"] == category_name, "question"]

chosen = my_categories[0]          # category of my first personalised entry
print(f"Questions in the '{chosen}' category (digit {last_two[0]} of my roll number):\n")
print(same_category(chosen, df).to_string())

Questions in the 'account' category (digit 4 of my roll number):

1                          how to reset password
4    how do i change my registered email address


## Q4: Add a keyword to one entry and save the DataFrame to CSV
I picked entry 0 (*what is the annual fee*).

In [6]:
ENTRY_INDEX = 0
print("Current keywords for:", df.loc[ENTRY_INDEX, "question"])
print(" ->", df.loc[ENTRY_INDEX, "keywords"])

new_keyword = input("\nEnter a new keyword to add: ").strip().lower()
df.loc[ENTRY_INDEX, "keywords"] = df.loc[ENTRY_INDEX, "keywords"] + " " + new_keyword
print("\nUpdated keywords:", df.loc[ENTRY_INDEX, "keywords"])

filename = f"{ROLL_NUMBER}_faq_data.csv"
df.to_csv(filename, index=False)
print("Saved to", filename)
df

Current keywords for: what is the annual fee
 -> fee cost price charge

Enter a new keyword to add: yearly

Updated keywords: fee cost price charge yearly
Saved to 1024160140_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge yearly,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i change my registered email address,Go to Profile > Account Settings > Email and confirm the change using the li...,email address update,account
5,how do i download my fee receipt,Open Payments > History and click Download next to the transaction.,receipt download invoice,billing


## Q5: Number of FAQ entries per category (`groupby`)

In [7]:
counts = df.groupby("category").size().reset_index(name="entries")
counts

,category,entries
0,account,2
1,billing,3
2,general,1


## Q6: Handling ties in the scoring function
`score_query_v2` reuses `score_query`, finds the highest score, and prints **all** entries that share it instead of silently choosing one.

In [8]:
def score_query_v2(query, df):
    results = score_query(query, df)
    if results.empty:
        print(f"Query: '{query}'  ->  no matching entry found.\n")
        return results

    top = results["score"].max()
    best = results[results["score"] == top]

    if len(best) > 1:
        print(f"Query: '{query}'  ->  TIE: {len(best)} entries share the top score of {top}:")
    else:
        print(f"Query: '{query}'  ->  single best match (score {top}):")
    print(best.to_string(index=False))
    print()
    return best

_ = score_query_v2("fee", df)               # tie: both "fee" entries
_ = score_query_v2("reset password", df)    # no tie

Query: 'fee'  ->  TIE: 2 entries share the top score of 3:
              question                                     answer category  score  confidence
what is the annual fee                  The annual fee is Rs 500.  billing      3         1.0
 how can i pay the fee You can pay via UPI, card, or net banking.  billing      3         1.0

Query: 'reset password'  ->  single best match (score 6):
             question                           answer category  score  confidence
how to reset password Go to Settings > Reset Password.  account      6         1.0

